# Análise Exploratória dos Dados - Booklog

## Integrantes

- Gabriel Nottoli Buck - RA 10425384 - 10425384@mackenzista.com.br
- Julia Andrade - RA 10427828 - 10427828@mackenzista.com.br
- João Vitor Rocha Miranda - RA 10427273 - 10427273@mackenzista.com.br

## Descrição

Este notebook realiza a análise exploratória e a preparação do conjunto de dados hipotético utilizado na N1 do projeto de recomendação personalizada de livros.

Os dados foram autorizados pelo professor como levantamento hipotético e não representam usuários reais.

## Histórico de alterações

| Data | Autor | Alteração |
|---|---|---|
| 15/09/2026 | Grupo | Criação da análise exploratória da N1 |

## 1. Carregamento

A base possui dois arquivos: catálogo de livros (`books.csv`) e avaliações (`ratings.csv`).

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

books = pd.read_csv(RAW / "books.csv")
ratings = pd.read_csv(RAW / "ratings.csv")

ratings["rating"] = pd.to_numeric(ratings["rating"], errors="raise")
ratings["interaction_date"] = pd.to_datetime(ratings["interaction_date"], errors="raise")

print("books:", books.shape)
print("ratings:", ratings.shape)
display(books.head())
display(ratings.head())

## 2. Qualidade e integridade dos dados

In [ ]:
quality = pd.DataFrame({
    "arquivo": ["books", "ratings"],
    "linhas": [len(books), len(ratings)],
    "valores_ausentes": [int(books.isna().sum().sum()), int(ratings.isna().sum().sum())],
    "linhas_duplicadas": [int(books.duplicated().sum()), int(ratings.duplicated().sum())],
})
display(quality)

print("book_id duplicados:", int(books["book_id"].duplicated().sum()))
print("pares usuário-livro duplicados:", int(ratings.duplicated(["user_id", "book_id"]).sum()))
print("notas fora da escala:", int((~ratings["rating"].between(1, 5)).sum()))
print("livros das avaliações ausentes do catálogo:", len(set(ratings["book_id"]) - set(books["book_id"])))

## 3. Estatísticas gerais

In [ ]:
n_users = ratings["user_id"].nunique()
n_books = books["book_id"].nunique()
n_ratings = len(ratings)
rated_books = ratings["book_id"].nunique()
sparsity = 1 - n_ratings / (n_users * rated_books)

summary = pd.Series({
    "usuários": n_users,
    "livros": n_books,
    "livros avaliados": rated_books,
    "avaliações": n_ratings,
    "nota média": ratings["rating"].mean(),
    "mediana": ratings["rating"].median(),
    "esparsidade": sparsity,
})
display(summary.to_frame("valor"))

## 4. Distribuição das notas

In [ ]:
rating_counts = ratings["rating"].value_counts().sort_index()
display(rating_counts.rename("quantidade").to_frame())

ax = rating_counts.plot(kind="bar", figsize=(8, 4), title="Distribuição das avaliações")
ax.set_xlabel("Nota")
ax.set_ylabel("Quantidade")
plt.tight_layout()
plt.show()

## 5. Interações por usuário

In [ ]:
ratings_per_user = ratings.groupby("user_id").size()
display(ratings_per_user.describe().to_frame("avaliações por usuário"))

ax = ratings_per_user.plot(kind="hist", bins=7, figsize=(8, 4), title="Avaliações por usuário")
ax.set_xlabel("Quantidade de avaliações")
plt.tight_layout()
plt.show()

## 6. Interações por livro

In [ ]:
ratings_per_book = ratings.groupby("book_id").size().sort_values(ascending=False)
top_books = (
    ratings_per_book.head(10)
    .rename("avaliações")
    .to_frame()
    .join(books.set_index("book_id")[["title", "genres"]])
)
display(top_books[["title", "genres", "avaliações"]])

ax = top_books.set_index("title")["avaliações"].sort_values().plot(
    kind="barh", figsize=(8, 5), title="Livros com mais avaliações"
)
ax.set_xlabel("Quantidade de avaliações")
ax.set_ylabel("Livro")
plt.tight_layout()
plt.show()

## 7. Avaliações por gênero

In [ ]:
ratings_genre = ratings.merge(books[["book_id", "genres"]], on="book_id", how="left")
genre_summary = (
    ratings_genre.groupby("genres")
    .agg(avaliações=("rating", "size"), nota_média=("rating", "mean"))
    .sort_values("avaliações", ascending=False)
)
display(genre_summary.round(2))

ax = genre_summary["avaliações"].plot(kind="bar", figsize=(8, 4), title="Avaliações por gênero")
ax.set_xlabel("Gênero")
ax.set_ylabel("Quantidade")
plt.tight_layout()
plt.show()

## 8. Esparsidade

In [ ]:
possible_pairs = n_users * rated_books
observed_pairs = len(ratings.drop_duplicates(["user_id", "book_id"]))
sparsity = 1 - observed_pairs / possible_pairs

print(f"Combinações possíveis: {possible_pairs}")
print(f"Combinações observadas: {observed_pairs}")
print(f"Esparsidade: {sparsity:.1%}")

A esparsidade indica que a maior parte das combinações usuário-livro não possui avaliação. Isso reforça a necessidade de avaliar com cuidado a técnica de recomendação usada na N2.

## 9. Preparação dos dados

In [ ]:
books_clean = (
    books.drop_duplicates()
    .sort_values("book_id")
    .reset_index(drop=True)
)

ratings_clean = (
    ratings.drop_duplicates()
    .sort_values(["user_id", "interaction_date", "book_id"])
    .reset_index(drop=True)
)

if not ratings_clean["rating"].between(1, 5).all():
    raise ValueError("Existem notas fora da escala de 1 a 5.")

if ratings_clean.duplicated(["user_id", "book_id"]).any():
    raise ValueError("Existem pares usuário-livro duplicados.")

if not set(ratings_clean["book_id"]).issubset(set(books_clean["book_id"])):
    raise ValueError("Existem avaliações de livros ausentes no catálogo.")

books_clean.to_csv(PROCESSED / "books_clean.csv", index=False)
ratings_clean.to_csv(
    PROCESSED / "ratings_clean.csv",
    index=False,
    date_format="%Y-%m-%d"
)

print("Arquivos tratados salvos em data/processed/.")

## 10. Síntese da N1

O conjunto hipotético contém **30 leitores, 36 livros e 345 avaliações**. Não foram encontrados valores ausentes, duplicidades ou notas fora da escala definida. A nota média é aproximadamente **3,62**, com mediana **3,5**. A matriz usuário x livro apresenta **68,1% de esparsidade**.

Esses resultados mostram um cenário coerente para iniciar a N2: existe histórico suficiente para testar uma estratégia personalizada, mas a base permanece esparsa, como é comum em sistemas de recomendação. Na próxima etapa, compararemos um baseline não personalizado com uma abordagem de recomendação adequada aos dados.